# Business Intelligence: PySpark + Power BI

**Pipeline:** Raw Data → PySpark ETL → Star Schema → Power BI → DAX → Dashboard → Insight

## 1. Mục tiêu
- Làm sạch dữ liệu lớn bằng PySpark.
- Xây dựng bảng Fact/Dimension.
- Tạo KPI phục vụ BI.
- Xuất dữ liệu cho Power BI.
- Thiết kế semantic model và dashboard.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName('TH06_BI').master('local[*]').getOrCreate()
print(spark.version)

## 2. Đọc dữ liệu

In [ ]:
sales = spark.read.option('header', True).option('inferSchema', True).csv('data/raw/sales_transactions_dirty.csv')
customers = spark.read.option('header', True).option('inferSchema', True).csv('data/raw/customers.csv')
products = spark.read.option('header', True).option('inferSchema', True).csv('data/raw/products.csv')
print('sales:', sales.count())
sales.printSchema()
sales.show(5, truncate=False)

## 3. Data Quality Audit

In [ ]:
from functools import reduce
nulls = sales.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in sales.columns])
nulls.show(truncate=False)
print('Duplicates:', sales.count() - sales.dropDuplicates().count())

### Bài tập 3.1
1. Xác định cột có missing.
2. Xác định duplicate.
3. Kiểm tra Revenue ≈ Quantity × UnitPrice × (1-Discount).
4. Đề xuất quy tắc cleaning và giải thích.

In [ ]:
sales_clean = (sales.dropDuplicates()
    .dropna(subset=['CustomerID','ProductID','Revenue'])
    .filter((F.col('Quantity') > 0) & (F.col('UnitPrice') > 0))
    .withColumn('Date', F.to_date('Date'))
    .withColumn('DateKey', F.date_format('Date','yyyyMMdd').cast('int')))
print(sales_clean.count())

## 4. Tạo KPI bằng PySpark

In [ ]:
kpi = sales_clean.agg(
    F.sum('Revenue').alias('TotalRevenue'),
    F.sum('Profit').alias('TotalProfit'),
    F.sum('Quantity').alias('UnitsSold'),
    F.countDistinct('OrderID').alias('TotalOrders'),
    F.countDistinct('CustomerID').alias('TotalCustomers')
)
kpi.show(truncate=False)

### Bài tập 4.1
Tính thêm: AOV, Profit Margin, Revenue per Customer và Return Rate.

## 5. Phân tích theo thời gian

In [ ]:
monthly = (sales_clean
    .withColumn('YearMonth', F.date_format('Date','yyyy-MM'))
    .groupBy('YearMonth')
    .agg(F.sum('Revenue').alias('Revenue'), F.sum('Profit').alias('Profit'), F.countDistinct('OrderID').alias('Orders'))
    .orderBy('YearMonth'))
monthly.show(15)

## 6. Xây dựng Star Schema

In [ ]:
dim_customer = customers.select('CustomerKey','CustomerID','CustomerSegment','Region','SignupDate').dropDuplicates(['CustomerKey'])
dim_product = products.select('ProductKey','ProductID','ProductName','Category','UnitCost','ListPrice').dropDuplicates(['ProductKey'])

dim_date = (sales_clean.select('Date').distinct()
    .withColumn('DateKey', F.date_format('Date','yyyyMMdd').cast('int'))
    .withColumn('Year', F.year('Date'))
    .withColumn('Quarter', F.concat(F.lit('Q'), F.quarter('Date')))
    .withColumn('MonthNo', F.month('Date'))
    .withColumn('Month', F.date_format('Date','MMM'))
    .withColumn('YearMonth', F.date_format('Date','yyyy-MM'))
    .withColumn('Weekday', F.date_format('Date','EEEE')))

fact = (sales_clean
    .join(dim_customer.select('CustomerKey','CustomerID'),'CustomerID','left')
    .join(dim_product.select('ProductKey','ProductID'),'ProductID','left')
    .select('TransactionID','OrderID','DateKey','CustomerKey','ProductKey','Channel','Quantity','UnitPrice','Discount','Revenue','Cost','Profit','IsReturn'))

print('Fact:', fact.count(), 'DimCustomer:', dim_customer.count(), 'DimProduct:', dim_product.count(), 'DimDate:', dim_date.count())

### Câu hỏi 6.1
- Grain của FactSales là gì?
- Vì sao ProductName không nên lặp lại trong FactSales?
- Relationship nào là 1:N?
- Vì sao Power BI thường phù hợp với star schema?

## 7. Xuất dữ liệu cho Power BI

In [ ]:
import os
os.makedirs('bi_output', exist_ok=True)

fact.coalesce(1).write.mode('overwrite').option('header',True).csv('bi_output/FactSales_csv')
dim_customer.coalesce(1).write.mode('overwrite').option('header',True).csv('bi_output/DimCustomer_csv')
dim_product.coalesce(1).write.mode('overwrite').option('header',True).csv('bi_output/DimProduct_csv')
dim_date.coalesce(1).write.mode('overwrite').option('header',True).csv('bi_output/DimDate_csv')

# Định dạng Parquet phù hợp cho pipeline dữ liệu lớn
fact.write.mode('overwrite').parquet('bi_output/FactSales_parquet')
dim_customer.write.mode('overwrite').parquet('bi_output/DimCustomer_parquet')
dim_product.write.mode('overwrite').parquet('bi_output/DimProduct_parquet')
dim_date.write.mode('overwrite').parquet('bi_output/DimDate_parquet')

## 8. Power BI Semantic Model
Trong Power BI Desktop:
1. Nạp các bảng Fact/Dim.
2. Tạo relationship many-to-one từ Fact sang Dimension.
3. Mark `DimDate` as Date Table.
4. Ẩn các khóa kỹ thuật khỏi Report view.
5. Tạo explicit measures bằng DAX.

## 9. DAX bắt buộc
Sao chép các measure trong `DAX_MEASURES.md`: Total Revenue, Total Profit, Total Orders, Total Customers, AOV, Profit Margin %, Revenue YoY %, Revenue YTD, Return Rate %.

## 10. Dashboard 1 – Executive Sales Overview
**Visual bắt buộc:**
- KPI cards: Revenue, Profit, Margin, Orders, Customers, AOV.
- Line chart: Revenue & Profit theo tháng.
- Bar chart: Revenue theo Category.
- Map/Bar: Revenue theo Region.
- Slicer: Year, Region, Channel, Category.

## 11. Dashboard 2 – Product & Customer Performance
- Top 10 sản phẩm theo Revenue/Profit.
- Revenue theo Customer Segment.
- Pareto sản phẩm hoặc khách hàng.
- Return Rate theo Category/Channel.
- Matrix: Category × Region.

## 12. Dashboard 3 – Management Insight
Sinh viên tự chọn 3–5 visual để trả lời tối thiểu 5 câu hỏi kinh doanh. Mỗi visual phải có kết luận ngắn và hành động đề xuất.

## 13. Bài tập phân tích
1. Kênh nào tăng trưởng nhanh nhất?
2. Nhóm sản phẩm nào doanh thu cao nhưng margin thấp?
3. Khu vực nào có Return Rate cao?
4. Top khách hàng có đóng góp bao nhiêu % doanh thu?
5. Doanh thu có tính mùa vụ không?
6. Có trade-off nào giữa tăng trưởng và lợi nhuận?

## 14. Performance & Big Data Discussion
So sánh hai kiến trúc:
- Raw CSV → Power BI xử lý toàn bộ.
- Raw Data → PySpark ETL → Fact/Dim → Power BI.

Thảo luận: volume, refresh time, reproducibility, governance, separation of concerns.

## 15. Executive Summary
Viết 5 insight định lượng + 3 recommendation. Mỗi insight phải nêu số liệu, visual hỗ trợ và tác động kinh doanh.

## 16. Sản phẩm nộp
- `.ipynb` đã chạy
- `.pbix`
- PDF/ảnh dashboard
- Báo cáo 2–4 trang
- thư mục `bi_output/` hoặc link dữ liệu
- README ngắn